In [2]:
import pandas as pd
import numpy as np
import re

# Load fresh
df = pd.read_parquet('../data/raw/storm_raw_combined.parquet')
df.shape

(1780730, 53)

In [3]:
# Drop >80% null cols
thresh = len(df) * 0.2
df = df.dropna(axis=1, thresh=int(thresh))


In [4]:
df.shape

(1780730, 44)

In [5]:
df.isnull().sum()

BEGIN_YEARMONTH            0
BEGIN_DAY                  0
BEGIN_TIME                 0
END_YEARMONTH              0
END_DAY                    0
END_TIME                   0
EPISODE_ID                 3
EVENT_ID                   0
STATE                      1
STATE_FIPS                 1
YEAR                       0
MONTH_NAME                 0
EVENT_TYPE                 0
CZ_TYPE                    0
CZ_FIPS                    0
CZ_NAME                    0
WFO                        3
BEGIN_DATE_TIME            0
CZ_TIMEZONE                0
END_DATE_TIME              0
INJURIES_DIRECT            0
INJURIES_INDIRECT          0
DEATHS_DIRECT              0
DEATHS_INDIRECT            0
DAMAGE_PROPERTY       612832
DAMAGE_CROPS          724876
SOURCE                113596
MAGNITUDE             863277
MAGNITUDE_TYPE       1236680
BEGIN_RANGE           899167
BEGIN_AZIMUTH         895287
BEGIN_LOCATION        672366
END_RANGE             895262
END_AZIMUTH           895561
END_LOCATION  

In [6]:
df['DAMAGE_PROPERTY'] = df['DAMAGE_PROPERTY'].fillna('0')
df['DAMAGE_CROPS']    = df['DAMAGE_CROPS'].fillna('0')
df['MAGNITUDE']       = df['MAGNITUDE'].fillna(0)

In [ ]:
df = df.drop(columns=[c for c in ['damage_crops_num','damage_total_num'] if c in df.columns]) # type: ignore


In [8]:
drop_cols = [
    'MAGNITUDE_TYPE', 'SOURCE',
    'BEGIN_LOCATION', 'END_LOCATION',
    'EPISODE_NARRATIVE', 'EVENT_NARRATIVE',
    'BEGIN_LAT', 'BEGIN_LON', 'END_LAT', 'END_LON',
    'BEGIN_RANGE', 'END_RANGE',
    'BEGIN_AZIMUTH', 'END_AZIMUTH',
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

In [9]:
df = df.dropna(subset=['EPISODE_ID', 'WFO', 'DATA_SOURCE', 'STATE'])

In [10]:
df.isnull().sum().sum()

np.int64(0)

In [12]:
print(repr(df['BEGIN_DATE_TIME'].iloc[0]))

'29-OCT-96 17:10:00'


In [13]:
fmt = '%d-%b-%y %H:%M:%S'

df['BEGIN_DATE_TIME'] = pd.to_datetime(df['BEGIN_DATE_TIME'], format=fmt, errors='coerce')
df['END_DATE_TIME']   = pd.to_datetime(df['END_DATE_TIME'],   format=fmt, errors='coerce')

print(repr(df['BEGIN_DATE_TIME'].iloc[0]))

df['duration_min'] = (
    (df['END_DATE_TIME'] - df['BEGIN_DATE_TIME'])
    .dt.total_seconds() / 60
).fillna(0).clip(0, 10000)

print(df['duration_min'].describe())

Timestamp('1996-10-29 17:10:00')
count    1.780726e+06
mean     9.878174e+02
std      2.404609e+03
min      0.000000e+00
25%      0.000000e+00
50%      6.000000e+00
75%      6.600000e+02
max      1.000000e+04
Name: duration_min, dtype: float64


In [14]:
def parse_damage(val):
    val = str(val).upper().strip()
    if val in ('0', '0K', '0.00K', '', 'NAN'): return 0.0
    m = re.match(r'([\d.]+)([KMB]?)', val)
    if not m: return 0.0
    n, s = float(m.group(1)), m.group(2)
    return n * {'K': 1e3, 'M': 1e6, 'B': 1e9}.get(s, 1)

def assign_tier(d):
    if d == 0:        return 0
    elif d < 10_000:  return 1
    elif d < 500_000: return 2
    else:             return 3

df['damage_property_num'] = df['DAMAGE_PROPERTY'].apply(parse_damage)
df['damage_crops_num']    = df['DAMAGE_CROPS'].apply(parse_damage)
df['damage_total_num']    = df['damage_property_num'] + df['damage_crops_num']
df['damage_tier']         = df['damage_property_num'].apply(assign_tier)

print(df['damage_tier'].value_counts())
print(df['damage_total_num'].describe())

damage_tier
0    1377651
1     205407
2     175438
3      22230
Name: count, dtype: int64
count    1.780726e+06
mean     3.556425e+05
std      3.162022e+07
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.790000e+10
Name: damage_total_num, dtype: float64


In [16]:
drop_more = [
    'BEGIN_DATE_TIME', 'END_DATE_TIME',  # replaced by duration_min
    'BEGIN_YEARMONTH', 'END_YEARMONTH',  # YEAR + MONTH_NAME enough
    'BEGIN_DAY', 'END_DAY',
    'BEGIN_TIME', 'END_TIME',
    'STATE_FIPS', 'CZ_FIPS',
    'CZ_NAME', 'CZ_TIMEZONE',
    'DATA_SOURCE', 'WFO', 'EPISODE_ID',
    'EVENT_ID', 'CZ_TYPE',
    'DAMAGE_PROPERTY', 'DAMAGE_CROPS',   # replaced by numeric cols
]

df = df.drop(columns=drop_more)

print(df.shape)
print(df.columns.tolist())


(1780726, 14)
['STATE', 'YEAR', 'MONTH_NAME', 'EVENT_TYPE', 'INJURIES_DIRECT', 'INJURIES_INDIRECT', 'DEATHS_DIRECT', 'DEATHS_INDIRECT', 'MAGNITUDE', 'duration_min', 'damage_property_num', 'damage_crops_num', 'damage_total_num', 'damage_tier']


In [17]:
print(df.isnull().sum().sum())
print(df.shape)

0
(1780726, 14)


In [18]:
df.to_parquet('../data/processed/storm_clean.parquet', index=False)
print("saved.")

saved.
